In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.appName("PortfolioValues").getOrCreate()

portfolio = spark.createDataFrame(
    [
        ("Alpha", "A", 1000),
        ("Alpha", "B", 2000),
        ("Beta", "A", 1500),
        ("Beta", "C", 2500),
        ("Gamma", "B", 1200),
        ("Gamma", "C", 1300),
    ],
    ["PE_firm", "company", "shares"],
)

prices = spark.createDataFrame(
    [
        ("2023-01-01", "A", 50.0),
        ("2023-01-01", "B", 20.0),
        ("2023-01-01", "C", 30.0),
        ("2023-01-02", "A", 52.0),
        ("2023-01-02", "B", 21.0),
        ("2023-01-02", "C", 31.0),
    ],
    ["date", "company", "closing_price"],
)

portfolio.show()
prices.show()

+-------+-------+------+
|PE_firm|company|shares|
+-------+-------+------+
|  Alpha|      A|  1000|
|  Alpha|      B|  2000|
|   Beta|      A|  1500|
|   Beta|      C|  2500|
|  Gamma|      B|  1200|
|  Gamma|      C|  1300|
+-------+-------+------+

+----------+-------+-------------+
|      date|company|closing_price|
+----------+-------+-------------+
|2023-01-01|      A|         50.0|
|2023-01-01|      B|         20.0|
|2023-01-01|      C|         30.0|
|2023-01-02|      A|         52.0|
|2023-01-02|      B|         21.0|
|2023-01-02|      C|         31.0|
+----------+-------+-------------+



In [ ]:
merged_df = portfolio.join(prices, on="company", how="inner")

# Compute portfolio value for each PE_firm and date
portfolio_value = merged_df.withColumn(
    "value",
    col("shares") * col("closing_price"),
)
portfolio_value.show()

In [ ]:
# Group by PE_firm and date and sum up the value
portfolio_value = portfolio_value.groupby(["PE_firm", "date"]).agg(
    sum("value").alias("portfolio_value")
)


portfolio_value.show()